# faiss db 실습

In [1]:
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
from langchain_community.document_loaders import PyPDFLoader

In [5]:
docs = PyPDFLoader("../../data/Sustainability_report_2024_kr.pdf").load()

In [6]:
len(docs)

83

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

rec_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=20,
)

chunk_docs = rec_splitter.split_documents(docs)
len(chunk_docs)

205

In [9]:
chunk_docs[0]

Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.1 (Macintosh)', 'creationdate': '2024-11-25T11:10:32+09:00', 'moddate': '2024-11-25T11:10:46+09:00', 'trapped': '/False', 'source': '../../data/Sustainability_report_2024_kr.pdf', 'total_pages': 83, 'page': 0, 'page_label': '1'}, page_content='A Journey Towards  \na Sustainable Future\n삼성전자 지속가능경영보고서 2024')

In [10]:
for item in chunk_docs:
    item.metadata = {**(item.metadata), "class":"wanted"}

In [11]:
chunk_docs[0]

Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.1 (Macintosh)', 'creationdate': '2024-11-25T11:10:32+09:00', 'moddate': '2024-11-25T11:10:46+09:00', 'trapped': '/False', 'source': '../../data/Sustainability_report_2024_kr.pdf', 'total_pages': 83, 'page': 0, 'page_label': '1', 'class': 'wanted'}, page_content='A Journey Towards  \na Sustainable Future\n삼성전자 지속가능경영보고서 2024')

# Faiss 벡터 DB 생성

In [12]:
from langchain_openai.embeddings import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [13]:
dim_size = len(embeddings.embed_query("삼성이 개발한 ai"))
print(dim_size)

3072


In [14]:
import faiss
from langchain.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

In [16]:
db = FAISS.from_documents(
    documents=[chunk_docs[0]],
    embedding = embeddings,
    ids= ["문서1"]
)

In [17]:
db.index_to_docstore_id

{0: '문서1'}

In [18]:
db = FAISS.from_documents(
    documents = [chunk_docs[0]],
    embedding = embeddings,
)
db.index_to_docstore_id

{0: '53debaad-d330-4bfe-a59b-85371342b44f'}

In [19]:
# db 확인 
db.docstore.__dict__["_dict"]

{'53debaad-d330-4bfe-a59b-85371342b44f': Document(id='53debaad-d330-4bfe-a59b-85371342b44f', metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.1 (Macintosh)', 'creationdate': '2024-11-25T11:10:32+09:00', 'moddate': '2024-11-25T11:10:46+09:00', 'trapped': '/False', 'source': '../../data/Sustainability_report_2024_kr.pdf', 'total_pages': 83, 'page': 0, 'page_label': '1', 'class': 'wanted'}, page_content='A Journey Towards  \na Sustainable Future\n삼성전자 지속가능경영보고서 2024')}

In [20]:
db.similarity_search("삼성", k=5)

[Document(id='53debaad-d330-4bfe-a59b-85371342b44f', metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.1 (Macintosh)', 'creationdate': '2024-11-25T11:10:32+09:00', 'moddate': '2024-11-25T11:10:46+09:00', 'trapped': '/False', 'source': '../../data/Sustainability_report_2024_kr.pdf', 'total_pages': 83, 'page': 0, 'page_label': '1', 'class': 'wanted'}, page_content='A Journey Towards  \na Sustainable Future\n삼성전자 지속가능경영보고서 2024')]

In [21]:
# db를 파일로 저장
vectorstore_db_path = "samsung_faiss.db"
index_name = "samsung2025"
db.save_local(
    folder_path = vectorstore_db_path,
    index_name=index_name
)


## 저장된 db 불러오기

In [22]:
load_db = FAISS.load_local(
    folder_path=vectorstore_db_path,
    index_name=index_name,
    embeddings=embeddings,
    allow_dangerous_deserialization=True # 피클 파일 형식인 db 불러올때 필요
)

In [23]:
load_db.docstore.__dict__["_dict"]

{'53debaad-d330-4bfe-a59b-85371342b44f': Document(id='53debaad-d330-4bfe-a59b-85371342b44f', metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.1 (Macintosh)', 'creationdate': '2024-11-25T11:10:32+09:00', 'moddate': '2024-11-25T11:10:46+09:00', 'trapped': '/False', 'source': '../../data/Sustainability_report_2024_kr.pdf', 'total_pages': 83, 'page': 0, 'page_label': '1', 'class': 'wanted'}, page_content='A Journey Towards  \na Sustainable Future\n삼성전자 지속가능경영보고서 2024')}

## 문서 추가하기

In [24]:
chunk_docs[1:10]

[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.1 (Macintosh)', 'creationdate': '2024-11-25T11:10:32+09:00', 'moddate': '2024-11-25T11:10:46+09:00', 'trapped': '/False', 'source': '../../data/Sustainability_report_2024_kr.pdf', 'total_pages': 83, 'page': 1, 'page_label': '2', 'class': 'wanted'}, page_content='A Journey Towards  \na Sustainable Future\n삼성전자 지속가능경영보고서 2024\nCEO 메시지\n회사 소개\n이해관계자 소통\nOur Company\n04\n05\n06\n준법과 윤리경영\nPrinciple\n53\n중대성 평가\nMateriality Assessment\n08\n임직원\n공급망\n사회공헌\n개인정보보호/보안\n고객의 안전/품질\nPeople\n31\n39\n45\n48\n50\n경제성과\n사회성과\n환경성과\n지역별 수자원 현황   \n사업부문별 환경성과\nFacts & Figures\n56\n57\n62\n65\n66\n독립된 인증인의 인증보고서\nScope 1, 2 온실가스 배출량 검증 의견서\nScope 3 온실가스 배출량 검증 의견서\nGRI Index\nTCFD 대조표\nSASB 대조표\n전사차원의 기후변화 대응 협력 활동\nAbout This Report\nAppendix\n70\n71\n72\n74\n77\n79\n81\n82\n[DX부문] \n추진체계 및 주요성과\n기후변화\n자원순환\n수자원 및 오염물질\n[DS부문]  \n추진체계 및 주요성과 \n기후변화\n수자원\n폐기물\n오염물질\nPlanet\n12\n13\n15\n17\n19\n20\n23\n26\n28\n삼성전자 지속

In [25]:
load_db.add_documents(
    chunk_docs[1:10]
)

['d77d704d-5841-4b6b-b531-e984329b4e42',
 '88afe95e-4dc0-4a55-9ebc-adf493de43cd',
 'efb7c552-9301-4713-9a87-60a69dd3f147',
 '8c11d068-deb7-42db-ab5b-6382d761eb72',
 'bf798e5a-10d5-42a8-b7ac-a634b99dd233',
 '00415e20-4119-4b00-9bff-5690994e4109',
 '6bc758e7-6dd9-4276-bdff-72eae095e915',
 '455031bb-1379-47f3-b3cb-de70cb213005',
 '45febaad-b199-4d71-bc68-d7252198cde8']

In [26]:
load_db.index_to_docstore_id

{0: '53debaad-d330-4bfe-a59b-85371342b44f',
 1: 'd77d704d-5841-4b6b-b531-e984329b4e42',
 2: '88afe95e-4dc0-4a55-9ebc-adf493de43cd',
 3: 'efb7c552-9301-4713-9a87-60a69dd3f147',
 4: '8c11d068-deb7-42db-ab5b-6382d761eb72',
 5: 'bf798e5a-10d5-42a8-b7ac-a634b99dd233',
 6: '00415e20-4119-4b00-9bff-5690994e4109',
 7: '6bc758e7-6dd9-4276-bdff-72eae095e915',
 8: '455031bb-1379-47f3-b3cb-de70cb213005',
 9: '45febaad-b199-4d71-bc68-d7252198cde8'}

In [27]:
vectorestore_db_path = "samsung_faiss.db"
index_name = "samsung2025"
load_db.save_local(
    folder_path=vectorestore_db_path,
    index_name=index_name
)


In [28]:
updated_db = FAISS.load_local(
    folder_path=vectorstore_db_path,
    index_name=index_name,
    embeddings=embeddings,
    allow_dangerous_deserialization=True # 피클 파일 형식인 db 불러올때 필요
)

updated_db.index_to_docstore_id

{0: '53debaad-d330-4bfe-a59b-85371342b44f',
 1: 'd77d704d-5841-4b6b-b531-e984329b4e42',
 2: '88afe95e-4dc0-4a55-9ebc-adf493de43cd',
 3: 'efb7c552-9301-4713-9a87-60a69dd3f147',
 4: '8c11d068-deb7-42db-ab5b-6382d761eb72',
 5: 'bf798e5a-10d5-42a8-b7ac-a634b99dd233',
 6: '00415e20-4119-4b00-9bff-5690994e4109',
 7: '6bc758e7-6dd9-4276-bdff-72eae095e915',
 8: '455031bb-1379-47f3-b3cb-de70cb213005',
 9: '45febaad-b199-4d71-bc68-d7252198cde8'}

## 문서 직접 추가하기

In [29]:
from langchain_core.documents import Document

# 직접 추가하기
updated_db.add_documents(
    [
        Document(
            page_content="새로운 문서는 이렇게 추가",
            metadata = {"source": "수동"}
        ), 
        Document(
            page_content="2024년 삼성 전자 주식 사지마세요",
            metadata = {"source": "윤택한"}
        ), 
    ]
)

['039212fd-7d74-4e47-a2b6-b2c680e69745',
 'ebfc231d-af95-48b0-a9e1-4defa729d6b6']

In [30]:
updated_db.index_to_docstore_id

{0: '53debaad-d330-4bfe-a59b-85371342b44f',
 1: 'd77d704d-5841-4b6b-b531-e984329b4e42',
 2: '88afe95e-4dc0-4a55-9ebc-adf493de43cd',
 3: 'efb7c552-9301-4713-9a87-60a69dd3f147',
 4: '8c11d068-deb7-42db-ab5b-6382d761eb72',
 5: 'bf798e5a-10d5-42a8-b7ac-a634b99dd233',
 6: '00415e20-4119-4b00-9bff-5690994e4109',
 7: '6bc758e7-6dd9-4276-bdff-72eae095e915',
 8: '455031bb-1379-47f3-b3cb-de70cb213005',
 9: '45febaad-b199-4d71-bc68-d7252198cde8',
 10: '039212fd-7d74-4e47-a2b6-b2c680e69745',
 11: 'ebfc231d-af95-48b0-a9e1-4defa729d6b6'}

In [31]:
updated_db.delete(['039212fd-7d74-4e47-a2b6-b2c680e69745'])
updated_db.index_to_docstore_id

{0: '53debaad-d330-4bfe-a59b-85371342b44f',
 1: 'd77d704d-5841-4b6b-b531-e984329b4e42',
 2: '88afe95e-4dc0-4a55-9ebc-adf493de43cd',
 3: 'efb7c552-9301-4713-9a87-60a69dd3f147',
 4: '8c11d068-deb7-42db-ab5b-6382d761eb72',
 5: 'bf798e5a-10d5-42a8-b7ac-a634b99dd233',
 6: '00415e20-4119-4b00-9bff-5690994e4109',
 7: '6bc758e7-6dd9-4276-bdff-72eae095e915',
 8: '455031bb-1379-47f3-b3cb-de70cb213005',
 9: '45febaad-b199-4d71-bc68-d7252198cde8',
 10: 'ebfc231d-af95-48b0-a9e1-4defa729d6b6'}

In [32]:
updated_db.similarity_search("삼성 전자 주식", k=5)

[Document(id='ebfc231d-af95-48b0-a9e1-4defa729d6b6', metadata={'source': '윤택한'}, page_content='2024년 삼성 전자 주식 사지마세요'),
 Document(id='00415e20-4119-4b00-9bff-5690994e4109', metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.1 (Macintosh)', 'creationdate': '2024-11-25T11:10:32+09:00', 'moddate': '2024-11-25T11:10:46+09:00', 'trapped': '/False', 'source': '../../data/Sustainability_report_2024_kr.pdf', 'total_pages': 83, 'page': 4, 'page_label': '5', 'class': 'wanted'}, page_content='삼성전자 지속가능경영보고서 2024\n05\nOur Company AppendixMateriality Assessment Facts & Figures PrinciplePlanet People\n회사소개\nAbout Us\n삼성전자주식회사(이하 삼성전자)는 인재와 기술을 바탕으로 최고의 제품과 서비스를 창출하여 인류사회에 공헌하는 글로벌 초일류기업을 지향합니다. \n이를 위해 삼성전자의 경영철학을 반영한 5가지 핵심가치 를 수립하였고, 핵심가치를 세부원칙과 행동지침 으로 구체화하여 삼성전자 임직원이 \n지켜야 할 글로벌 행동규범(Global Code of Conduct) 을 제정하였습니다. 삼성전자는 조직문화에 5가지 핵심가치를 내재화하고 글로벌 행동규범을 \n모든 경영활동의 기준으로 삼아 지속적으로 성장해갈 것입니다. \n사업부문 및 글로벌 네트워크 소개\n삼성전자는 제품 특성에 따라 DX(Device eXperience)와 DS(Device Solutions

## 벡터 스토어 합치기
- 물리적 합치기
- 검색기만 하이브리드로 사용

In [33]:
# 2024년 데이터를 앞에서 10개 -> db1 
# 2024년 데이터를 11~20 -> db2
# 이 두 벡터 스토어를 합쳐서 -> db3 

In [34]:
db1 = FAISS.from_documents(
    chunk_docs[0:10],
    embedding=embeddings
)

db2 = FAISS.from_documents(
    chunk_docs[10:20],
    embedding=embeddings
)

In [35]:
# 1. 새로운 공간에 합친 데이터베이스를 만들기 -> db3
# 2. db1 에 db2 를 합쳐버리기

In [36]:
db1.merge_from(
    target=db2
)
db1.index_to_docstore_id

{0: 'd9f1d4ed-bcfa-4f0e-82f0-79a218ad0505',
 1: '9a97ad8f-d703-4fb5-86e2-518bf89e9f94',
 2: '10280b8c-f437-4006-88a0-5768abcb97f4',
 3: '2b909b5e-eddd-4bf9-a77a-53159f4eb8b3',
 4: '0eb2e57a-2eaa-43d9-aab1-71d711fad64a',
 5: '195aa732-39b4-4a52-981f-b37f2c4351ae',
 6: '2e03960f-22e8-463f-8fb3-732f0856f420',
 7: '95f2ecc0-f47c-4cd2-ab83-4e1f39a4e5a4',
 8: 'fd772ff0-06f1-43ff-a7b1-7a2a37fee241',
 9: '6e3cdade-68f1-420f-818c-f813066ead7f',
 10: 'ed59c4ad-2c72-4164-929d-cfd48feff822',
 11: '17604010-7846-4cfd-af89-1e0d57ec712a',
 12: '3e07a875-292d-48e4-be3c-2df74b2c57f9',
 13: 'eccc03ab-ef00-4480-8153-3f68887a37c8',
 14: '92e25a90-cd6d-4f2f-81e8-3649cab5a119',
 15: '6c019e03-7350-4c20-87d1-5efc9bb2b178',
 16: 'b80b162e-98c3-4d7b-a3ca-3f4a9f2b909a',
 17: '78e62384-1f40-465e-ab2e-9690e4e3e937',
 18: 'bb1a3a74-f634-4c5c-b16b-0a81949efcbf',
 19: 'acc1f1b0-2633-4808-bede-d8d084ed7908'}

In [37]:
db2.index_to_docstore_id

{0: 'ed59c4ad-2c72-4164-929d-cfd48feff822',
 1: '17604010-7846-4cfd-af89-1e0d57ec712a',
 2: '3e07a875-292d-48e4-be3c-2df74b2c57f9',
 3: 'eccc03ab-ef00-4480-8153-3f68887a37c8',
 4: '92e25a90-cd6d-4f2f-81e8-3649cab5a119',
 5: '6c019e03-7350-4c20-87d1-5efc9bb2b178',
 6: 'b80b162e-98c3-4d7b-a3ca-3f4a9f2b909a',
 7: '78e62384-1f40-465e-ab2e-9690e4e3e937',
 8: 'bb1a3a74-f634-4c5c-b16b-0a81949efcbf',
 9: 'acc1f1b0-2633-4808-bede-d8d084ed7908'}

In [44]:
# 만약에 나는 전혀 다른 db3 이라는 빈 db에 데이터를 합치고 싶어
db3 = FAISS(
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
    embedding_function=embeddings,
    index = faiss.IndexFlatL2(dim_size)
)

In [45]:
db3.index_to_docstore_id

{}

In [46]:
db3.merge_from(
    target=db2
)
db3.index_to_docstore_id

{0: 'ed59c4ad-2c72-4164-929d-cfd48feff822',
 1: '17604010-7846-4cfd-af89-1e0d57ec712a',
 2: '3e07a875-292d-48e4-be3c-2df74b2c57f9',
 3: 'eccc03ab-ef00-4480-8153-3f68887a37c8',
 4: '92e25a90-cd6d-4f2f-81e8-3649cab5a119',
 5: '6c019e03-7350-4c20-87d1-5efc9bb2b178',
 6: 'b80b162e-98c3-4d7b-a3ca-3f4a9f2b909a',
 7: '78e62384-1f40-465e-ab2e-9690e4e3e937',
 8: 'bb1a3a74-f634-4c5c-b16b-0a81949efcbf',
 9: 'acc1f1b0-2633-4808-bede-d8d084ed7908'}

In [47]:
# id 겹쳐서 아래는 실행 안됨

# db3.merge_from(
#     target=db1
# )

# "samsung2025::5aa19b24-ee08-419f-a766-8666d44a3d9c" 이런식으로 인덱스 앞에 메타정보 필요